In [1]:
nr_std = 2

In [33]:
import pandas as pd
import numpy as np

# Carro dålig 

users = ["Ebba 3", "Fabian", "Daniel", "Daniel 53", "Viktor", "Olle", "John", "Erik 52", "Jan 50", "Fredrik 37"]

dfs = []

for user in users:
    
    df = pd.read_csv(f"Garmin data\\Activities {user}.csv")
    
    df = df[df["Aktivitetstyp"].isin(["Löpning", "Terränglöpning", "Löpband"])]
    
    df["Löpare"] = user
    
    df["Tid"] = pd.to_timedelta(df["Tid"]).dt.total_seconds()

    if "Medelfart" in df.columns:
        df["Medeltempo"] = df["Medelfart"]

    # tar bort tomma kolumner, problematiskt för viss löpare (Olle)
    df = df[(df["Medeltempo"] != "--") & (df["Medelpuls"] != "--") & (df["Total stigning"] != "--") & (df["Totalt nedför"] != "--")]

    df["Medeltempo_s"] = pd.to_timedelta("00:" + df["Medeltempo"]).dt.total_seconds()
    
    # formattera kolumnvärden så att de är float
    numeric_cols = ["Medelpuls", "Maxpuls", "Total stigning", "Totalt nedför", "Distans"]
    df[numeric_cols] = df[numeric_cols].apply(lambda s: pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce"))
    
    df = df[["Löpare", "Datum", "Distans", "Tid", "Medeltempo", "Medeltempo_s", "Medelpuls", "Maxpuls", "Total stigning", "Totalt nedför"]]
    
    # Vi kollar på 'Medeltempo_s' då det oftast är där extrema GPS-fel syns
    mean_tempo = df["Medeltempo_s"].mean()
    std_tempo = df["Medeltempo_s"].std()
    
    # Behåll bara rader som ligger inom ±2 standardavvikelser
    upper_limit = mean_tempo + nr_std * std_tempo
    lower_limit = mean_tempo - nr_std * std_tempo
    
    # Vi printar ut hur mycket som rensas så du har koll!
    pushed_out = df[(df["Medeltempo_s"] > upper_limit) | (df["Medeltempo_s"] < lower_limit)]
    print(f"{user}: Tog bort {len(pushed_out)} avvikande pass.")
    
    df = df[(df["Medeltempo_s"] <= upper_limit) & (df["Medeltempo_s"] >= lower_limit)]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["Duration_h"] = df["Tid"] / 3600

max_hr = df.groupby("Löpare")["Maxpuls"].transform("max")

hr_threshold = 0.95 * max_hr

df["IF"] = df["Medelpuls"] / hr_threshold

df["TSS"] = df["Duration_h"] * (df["IF"]**2) * 100

df["TSS_per_km"] = df["TSS"] / df["Distans"]

df["Stigning_per_km"] = df["Total stigning"] / df["Distans"]
df["Nedför_per_km"] = df["Totalt nedför"] / df["Distans"]

df["Datum"] = pd.to_datetime(df["Datum"], dayfirst=False, format="mixed")

# 2. Hitta max-datumet PER LÖPARE och mappa tillbaka det till varje rad
# transform('max') ser till att resultatet har samma längd som ursprungliga df
df['Max_Datum_User'] = df.groupby('Löpare')['Datum'].transform('max')

# 3. Räkna ut dagar sedan det senaste passet (per löpare)
df['Dagar sedan'] = (df['Max_Datum_User'] - df['Datum']).dt.days

# 4. Räkna ut den exponentiella vikten
decay_rate = 0.005
df['Vikt exp'] = np.exp(-decay_rate * df['Dagar sedan'])


# (Valfritt) Ta bort hjälpkolumnen för max-datum
#df = df.drop(columns=['Max_Datum_User'])


Ebba 3: Tog bort 2 avvikande pass.
Fabian: Tog bort 0 avvikande pass.
Daniel: Tog bort 2 avvikande pass.
Daniel 53: Tog bort 0 avvikande pass.
Viktor: Tog bort 8 avvikande pass.
Olle: Tog bort 4 avvikande pass.
John: Tog bort 4 avvikande pass.
Erik 52: Tog bort 1 avvikande pass.
Jan 50: Tog bort 24 avvikande pass.
Fredrik 37: Tog bort 4 avvikande pass.


In [34]:
# visar att AR1 bästa covariance-strukturen men använder inte vikter
import statsmodels.api as sm
import statsmodels.formula.api as smf

cov_structs = {
    "Independence": sm.cov_struct.Independence(),
    "Exchangeable": sm.cov_struct.Exchangeable(),
    "AR1": sm.cov_struct.Autoregressive(grid=True)
}

for name, cov in cov_structs.items():
    model = smf.gee(
        "Medeltempo_s ~ Stigning_per_km + Nedför_per_km + IF",
        groups=df["Löpare"],
        data=df,
        weights=df["Vikt exp"],
        cov_struct=cov
    )
    result = model.fit()
    qic, qicu = result.qic()
    print(f"{name}: QIC = {qic:.2f}, QICu = {qicu:.2f}")

Independence: QIC = 2131.72, QICu = 1828.14
Exchangeable: QIC = 2400.33, QICu = 1776.21
AR1: QIC = 2083.33, QICu = 1766.41


c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_estimating_equations.py:1934: UserWarning: QIC values obtained using scale=None are not appropriate for comparing models
  warnings.warn("QIC values obtained using scale=None are not "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\generalized_estimating_equations.py:1934: UserWarning: QIC values obtained using scale=None are not appropriate for comparing models
  warnings.warn("QIC values obtained using scale=None are not "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: weights not implemented for autoregressive cov_struct, using unweighted covariance estimate
  warnings.warn("weights not implemented for autoregressive "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: 

In [36]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# GEE påminner mycket om MixedLM men har en 'weights'-parameter
model_gee = smf.gee("Medeltempo_s ~ Stigning_per_km + Nedför_per_km + IF", 
                    groups=df["Löpare"], 
                    data=df, 
                    weights=df["Vikt exp"], # Här petar vi in dina tidsvikter!
                    cov_struct=sm.cov_struct.Autoregressive(grid=True))

result_gee = model_gee.fit()
print(result_gee.summary())

                               GEE Regression Results                              
Dep. Variable:                Medeltempo_s   No. Observations:                 1730
Model:                                 GEE   No. clusters:                       10
Method:                        Generalized   Min. cluster size:                  16
                      Estimating Equations   Max. cluster size:                 684
Family:                           Gaussian   Mean cluster size:               173.0
Dependence structure:       Autoregressive   Num. iterations:                    20
Date:                     Thu, 09 Apr 2026   Scale:                        2795.407
Covariance type:                    robust   Time:                         15:15:53
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         551.5705     38.760     14.231      0.000     475.603     

c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: weights not implemented for autoregressive cov_struct, using unweighted covariance estimate
  warnings.warn("weights not implemented for autoregressive "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: weights not implemented for autoregressive cov_struct, using unweighted covariance estimate
  warnings.warn("weights not implemented for autoregressive "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: weights not implemented for autoregressive cov_struct, using unweighted covariance estimate
  warnings.warn("weights not implemented for autoregressive "
c:\Users\ebbat\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\cov_struct.py:811: NotImplementedWarning: we

In [37]:
# 1. Hämta de färdiga prediktionerna direkt från modellen
df['pred_population'] = result_gee.fittedvalues

# 2. Räkna ut residualen (skillnaden mellan verklighet och gissning)
df['residual'] = df['Medeltempo_s'] - df['pred_population']

# 3. Beräkna löparens personliga offset
# Vi grupperar på löpare och tar ett viktat medelvärde av deras residualer
runner_offsets = df.groupby('Löpare').apply(
    lambda x: np.average(x['residual'], weights=x['Vikt exp'])
).to_dict()

In [38]:
def format_tempo(seconds):
    if pd.isna(seconds): return "N/A"
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes}:{sec:02d}/km"

def predict_safe_gee(runner_name, if_val, stigning, nedfor, model, offsets):
    # GEE använder .params för de globala koefficienterna
    p = model.params
    
    # Beräkna det generella tempot (Population Average)
    base_tempo = (p['Intercept'] + 
                  p['IF'] * if_val + 
                  p['Stigning_per_km'] * stigning + 
                  p['Nedför_per_km'] * nedfor)
    
    # Hämta löparens specifika offset som vi räknade ut ovan
    runner_offset = offsets.get(runner_name, 0)
        
    final_seconds = base_tempo + runner_offset
    return final_seconds, runner_offset

def predict_tempo_for_all_gee(df, model, offsets, if_val=0.85, stigning=0, nedfor=0):
    unika_löpare = df['Löpare'].unique()

    print(f"Predikterat tempo (GEE) för IF {if_val}, stigning={stigning}, nedför={nedfor}")
    print(f"{'Löpare':<15} | {'Tempo (IF)':<15} | {'Offset (sek)':<12}")
    print("-" * 50)

    ranking = []

    for namn in unika_löpare:
        seconds, offset = predict_safe_gee(namn, if_val, stigning, nedfor, model, offsets)
        tempo_str = format_tempo(seconds)
        
        ranking.append({'Namn': namn, 'Sekunder': seconds, 'Tempo': tempo_str, 'Offset': offset})
        print(f"{namn:<15} | {tempo_str:<15} | {offset:>10.1f}s")

    print("\n--- Snabbhetsranking (Standardiserad runda) ---")
    # Sortera på Sekunder istället för strängen för att få korrekt ordning
    ranking_df = pd.DataFrame(ranking).sort_values('Sekunder')
    print(ranking_df[['Namn', 'Tempo', 'Offset']].to_string(index=False))


predict_tempo_for_all_gee(df, result_gee, runner_offsets, if_val=0.85, stigning=0, nedfor=0)

Predikterat tempo (GEE) för IF 0.85, stigning=0, nedför=0
Löpare          | Tempo (IF)      | Offset (sek)
--------------------------------------------------
Ebba 3          | 5:54/km         |       59.6s
Fabian          | 6:13/km         |       78.6s
Daniel          | 5:55/km         |       60.7s
Daniel 53       | 5:02/km         |        7.1s
Viktor          | 6:02/km         |       67.5s
Olle            | 4:26/km         |      -28.4s
John            | 4:28/km         |      -26.5s
Erik 52         | 5:04/km         |        9.5s
Jan 50          | 4:57/km         |        2.7s
Fredrik 37      | 4:15/km         |      -40.1s

--- Snabbhetsranking (Standardiserad runda) ---
      Namn   Tempo     Offset
Fredrik 37 4:15/km -40.110421
      Olle 4:26/km -28.380656
      John 4:28/km -26.457450
    Jan 50 4:57/km   2.746194
 Daniel 53 5:02/km   7.087680
   Erik 52 5:04/km   9.496869
    Ebba 3 5:54/km  59.599417
    Daniel 5:55/km  60.669021
    Viktor 6:02/km  67.477308
    Fabian 6:

In [39]:
import numpy as np
import pandas as pd

def validate_model_gee(model, df):
    # 1. Skapa generella prediktioner (Population Average) från modellen
    df['pred_population'] = model.fittedvalues

    # Räkna ut den generella residualen (skillnaden mellan verklighet och snittet)
    df['residual'] = df['Medeltempo_s'] - df['pred_population']

    # Beräkna varje löpares personliga offset (viktat medelvärde av deras residualer)
    # Nyare pass (högre Vikt exp) väger tyngre i bedömningen av deras snabbhet
    runner_offsets = df.groupby('Löpare').apply(
        lambda x: np.average(x['residual'], weights=x['Vikt exp'])
    ).to_dict()

    # Lägg ihop den generella prediktionen med löparens personliga offset
    df['Predikterat_s'] = df['pred_population'] + df['Löpare'].map(runner_offsets)
    
    # Räkna ut hur många sekunder fel modellen gissade på varje enskilt pass
    df['Fel_sekunder'] = (df['Medeltempo_s'] - df['Predikterat_s']).abs()

    # 2. Gruppera per löpare för att se statistiken
    # Vi använder namngiven aggregering för att det är tydligare i Pandas
    validering = df.groupby('Löpare').agg(
        Antal_pass=('Medeltempo_s', 'count'),
        Snittfel_sek=('Fel_sekunder', 'mean')
    )

    # 3. Sortera och visa
    validering = validering.sort_values('Snittfel_sek')
    print("Validering per löpare (Snittfel i sekunder per km)")
    print("-" * 50)
    print(validering)

    # 4. Räkna ut VIKTAT R-squared
    # Vi måste bedöma modellens R2 baserat på samma vikter som den tränades på
    weights = df['Vikt exp']
    
    # Viktat genomsnitt av hela datamängden
    mean_weighted = np.average(df['Medeltempo_s'], weights=weights)
    
    # SS_res: Summan av de kvadrerade felen (viktade)
    ss_res_w = (weights * (df['Medeltempo_s'] - df['Predikterat_s']) ** 2).sum()
    
    # SS_tot: Totala variationen från snittet (viktad)
    ss_tot_w = (weights * (df['Medeltempo_s'] - mean_weighted) ** 2).sum()
    
    # Formeln för R2
    r2_weighted = 1 - (ss_res_w / ss_tot_w)

    print("-" * 50)
    print(f"Viktad förklaringsgrad (R2): {r2_weighted:.2f}")

# Kör funktionen med ditt resultat-objekt
validate_model_gee(result_gee, df)

Validering per löpare (Snittfel i sekunder per km)
--------------------------------------------------
            Antal_pass  Snittfel_sek
Löpare                              
Daniel              25     16.997680
Erik 52             76     18.616924
Jan 50             684     20.995892
Daniel 53           16     23.172988
Fabian              20     24.186220
Viktor             182     24.682024
Ebba 3              63     26.356810
Fredrik 37         442     27.330095
John               161     44.144533
Olle                61     75.631822
--------------------------------------------------
Viktad förklaringsgrad (R2): 0.61


In [19]:
# 1. Skapa den totala prediktionen (Bas + Personlig offset)
df['Pred_Total'] = result_gee.fittedvalues + df['Löpare'].map(runner_offsets)

# 2. Beräkna absolut fel per pass
df['Fel_sekunder'] = (df['Medeltempo_s'] - df['Pred_Total']).abs()

# 3. Sortera och plocka ut de 5 sista passen per löpare
df['Datum'] = pd.to_datetime(df['Datum']) # Säkerställ datumformat
recent_gee = df.sort_values('Datum').groupby('Löpare').tail(5)

# 4. Skapa tabellen
tabell_gee = recent_gee.groupby('Löpare').agg(
    Antal_pass=('Medeltempo_s', 'count'),
    Snittfel_Senaste_5=('Fel_sekunder', 'mean')
).sort_values('Snittfel_Senaste_5')

print("--- VALIDERING: GEE (Tidsviktad) - ENDAST 5 SENASTE PASSEN ---")
print(tabell_gee)
print(f"\nTotalt genomsnittsfel för alla: {tabell_gee['Snittfel_Senaste_5'].mean():.2f}s")

--- VALIDERING: GEE (Tidsviktad) - ENDAST 5 SENASTE PASSEN ---
            Antal_pass  Snittfel_Senaste_5
Löpare                                    
Jan 50               5            8.580661
Viktor               5            9.108144
Olle                 5           11.674641
Fabian               5           17.293940
Fredrik 37           5           17.319774
Erik 52              5           17.915164
Daniel 53            5           24.093366
Daniel               5           24.255461
Ebba 3               5           25.715052
John                 5           46.382029

Totalt genomsnittsfel för alla: 20.23s
